# FBS Win Probability — Initial Demo

This notebook runs the whole pipeline one step at a time: it pulls the upcoming 2026 Week 5 schedule from the
College Football Data API, stores the data in our private Amazon S3 bucket (cloud file storage), builds the
training data, downloads it back from S3, trains a model, and predicts who wins each Week 5 game.

**How to run:** choose **Kernel → Restart Kernel and Clear All Outputs**, then run each cell from top to bottom
with **Shift+Enter**. Every step calls the same tested pipeline functions as the command-line tools
(`src.pipeline.week_refresh` and `src.pipeline.s3_demo`).

## Step 0 — Setup and safety checks

**What this does:** sets the options for this run, then checks that we can reach AWS (Amazon's cloud), that our
S3 bucket is private, and that the CFBD API key (the password-like token for the data service) is present.

**Why it matters:** the run stops here, before any data is requested, if anything is misconfigured. No keys,
passwords, or the bucket's real name are ever shown.

In [ ]:
# Run from the repository root so `src` imports work when JupyterLab opens this notebook from notebooks/.
import os
from pathlib import Path

_root = Path.cwd()
while not (_root / "src" / "pipeline").is_dir() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)

from datetime import datetime, timezone

# ---- Settings for this run -------------------------------------------------------------------
SEASON = 2026
WEEK = 5
RUN_TAG = f"nb-week{WEEK}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"  # New folder names for every run.
HYPOTHETICAL = (0.2, 7, 1)          # (win % diff, points-margin diff, home field) for Step 8b.
USE_EXISTING_DATASET_ID = None      # Fallback: set to a dataset ID already in S3 to skip Steps 1–5.
SHOW_BUCKET_NAME = False            # False shows the bucket as "<private project bucket>".
# ----------------------------------------------------------------------------------------------

import csv
import io

from src.features import dataset, games
from src.ingestion import cfbd, s3_storage
from src.ingestion.storage import parse_games, save_snapshot
from src.models.baseline import PREDICTORS, check_status, load_trusted_model, read_file, read_json, train_baseline
from src.models.predict import predict_saved_sample, predict_vector
from src.pipeline import notebook_views as views
from src.pipeline import s3_demo, week_refresh

views.reset()
views.say(f"**Run tag:** `{RUN_TAG}`")
with views.step(0, "setup"):
    if SEASON != week_refresh.SEASON:
        raise week_refresh.RefreshError("This demo supports the 2026 season only.")
    week_refresh.run_tag(RUN_TAG)
    RUN_DIRS = week_refresh.output_dirs(week_refresh.PROJECT_ROOT, RUN_TAG)
    for _path in RUN_DIRS.values():
        week_refresh.check_new_dir(_path)          # Never reuse or overwrite an earlier run's folders.
    NORMALIZED_ROOT = RUN_DIRS["interim"] / "games"
    FEATURES_DIR = RUN_DIRS["features"]
    RUN_DIR = RUN_DIRS["run"]
    _settings = s3_storage.configuration()          # Nonsecret bucket/prefix/region names only.
    store = s3_storage.S3Store(None, _settings["FBS_S3_BUCKET"], _settings["FBS_S3_PREFIX"])
    BUCKET = views.bucket_label(store.bucket, SHOW_BUCKET_NAME)
    store.client = s3_storage.make_client(_settings["FBS_S3_REGION"], _settings["FBS_AWS_PROFILE"])
    store._private_bucket()                        # Read-only check of all four Block Public Access settings.
    _checks = ["**AWS connection:** OK", "**Bucket is private:** all 4 public-access blocks on",
               f"**Bucket:** {views.esc(BUCKET)}, folder: `{store.prefix}/`"]
    if USE_EXISTING_DATASET_ID:
        _checks.append("**CFBD API key:** not needed (using an existing dataset)")
    else:
        cfbd._api_key()                            # Presence check only; the value is never shown.
        _checks.append("**CFBD API key:** found")
        HISTORY = week_refresh.verify_history(*week_refresh.discover_history(week_refresh.PROJECT_ROOT))
        _checks.append(f"**History:** {len(HISTORY)} past seasons (2021–2025) found and fingerprint-verified")
    views.say("\n".join(_checks))

## Step 1 — Ingest the Week 5 schedule from the API

**What this does:** makes one request to the College Football Data API (a public college football data service)
for the 2026 regular-season Week 5 games, and saves the response exactly as received.

**Why it matters:** this is the "data fed from an API" part of the pipeline. These games have not been played yet,
so there are no scores in this data.

In [ ]:
if views.live_step(1, USE_EXISTING_DATASET_ID):
    with views.step(1, "cfbd_fetch"):
        fetched = cfbd.fetch_games(SEASON, WEEK)
        raw_path, manifest_path = save_snapshot(fetched, RUN_DIRS["raw"] / "games")
        records = parse_games(fetched.raw_response)
        if not records:
            raise week_refresh.RefreshError("CFBD returned no games for this week; nothing uploaded.")
        views.say(views.fetch_line(fetched.http_status, len(records), fetched.retrieved_at))
        views.show(views.games_table(records))

## Step 2 — Store the raw data in S3

**What this does:** uploads the raw API response and its manifest to our private S3 bucket. A **manifest** is a
small receipt saved with the raw data: it records when the data was pulled, what was requested, how many games
came back, and a fingerprint (hash) proving the file hasn't changed.

**Why it matters:** the original data is kept safely in the cloud. Uploads never overwrite an existing file, and
each upload is read back and compared byte for byte.

In [ ]:
if views.live_step(2, USE_EXISTING_DATASET_ID):
    with views.step(2, "s3_upload_raw"):
        raw_objects = store.upload_raw(manifest_path)
        sizes = {path.name: path.stat().st_size for path in (raw_path, manifest_path)}
        views.show(views.raw_upload_table(raw_objects, sizes, SHOW_BUCKET_NAME))

## Step 3 — Clean the games

**What this does:** checks every game record and keeps only FBS vs FBS games (FBS is the top division of college
football). Games against FCS schools (the division below) are excluded, and games without an announced kickoff
time are set aside instead of guessed.

**Why it matters:** bad or out-of-scope records never reach the model, and every removed game is listed with a
reason, so nothing is dropped silently.

In [ ]:
if views.live_step(3, USE_EXISTING_DATASET_ID):
    with views.step(3, "normalize"):
        new_normalized = games.normalize_snapshot(manifest_path, NORMALIZED_ROOT)
        for _path in HISTORY:                      # build_dataset reads all seasons from one folder.
            games.normalize_snapshot(_path, NORMALIZED_ROOT)
        quality = read_json(read_file(new_normalized / "quality_report.json"))
        cleaning = read_json(read_file(new_normalized / "exclusions.json"))
        views.show(views.cleaning_counts(quality["counts"]))
        views.show(views.set_aside_table(cleaning, records))

## Step 4 — Features: what the model looks at

**What this does:** builds the dataset, which computes three features (inputs to the model) for every game from
the previous season's results, then shows one real Week 5 matchup.

**Why it matters:** the model sees only these three numbers. Final scores are used only as the training label
("did the home team win?"), never as an input, and upcoming games have no label at all.

In [ ]:
views.show(views.feature_table())
if views.live_step(4, USE_EXISTING_DATASET_ID, what="the example matchup"):
    with views.step(4, "build_dataset"):
        report = dataset.build_dataset([*HISTORY, manifest_path], FEATURES_DIR, allow_retrospective_demo=True,
                                       normalized_root=NORMALIZED_ROOT)
        check_status(report)                       # Retrospective-demo flags must be unchanged.
        prediction_rows = list(csv.DictReader(io.StringIO(read_file(FEATURES_DIR / "prediction.csv").decode("utf-8"))))
        heading, example = views.example_matchup(prediction_rows[0])
        views.say(heading)
        views.show(example)

## Step 5 — Build the training data and store it in S3

**What this does:** shows how the games are split — past seasons to learn from, one season to test on, and the
upcoming week to predict — then uploads the six dataset files to S3.

**Why it matters:** the model is tested only on games it never trained on. The dataset gets a fingerprint (an ID
computed from its contents); the next step uses it automatically, so the presenter never types it.

In [ ]:
if views.live_step(5, USE_EXISTING_DATASET_ID):
    with views.step(5, "s3_upload_dataset"):
        views.show(views.split_table(report, WEEK))
        dataset_objects = store.upload_dataset(FEATURES_DIR, allow_retrospective_demo=True)
        views.show(views.dataset_upload_table(dataset_objects, SHOW_BUCKET_NAME))
        DATASET_ID = report["dataset_id"]
        views.say(f"**Dataset fingerprint:** `{views.fingerprint(DATASET_ID)}` (the next cell uses it automatically)")
else:
    DATASET_ID = USE_EXISTING_DATASET_ID
    views.say(f"**Dataset fingerprint:** `{views.fingerprint(DATASET_ID)}` (existing dataset; the next cell uses it "
              "automatically)")

## Step 6 — Load the training data from S3

**What this does:** downloads the six dataset files from S3 and checks each file's fingerprint (a SHA-256 hash,
a unique code computed from the file's contents) before anything uses it.

**Why it matters:** the model trains on exactly the data stored in the cloud. If any file were changed or
incomplete, the run would stop here.

In [ ]:
with views.step(6, "s3_download"):
    download = store.download_dataset(DATASET_ID, RUN_DIR / "dataset", allow_retrospective_demo=True)
    views.show(views.download_table(download))
    views.say("Model will train on the copy downloaded from S3 (downloaded and verified, **not streamed**).")

## Step 7 — Train and evaluate the model

**What this does:** trains a logistic regression (a standard model that outputs a probability) on the 2022–2024
games from the downloaded copy, then scores it on the 2025 games it never saw.

**Why it matters:** we compare it with a simple baseline that always predicts the historical home-win rate. Brier
score and log loss measure how good the probabilities are (lower is better); accuracy is how often the favored
team won.

In [ ]:
with views.step(7, "local_training"):
    metadata, evaluation = train_baseline(RUN_DIR / "dataset", RUN_DIR / "model", allow_retrospective_demo=True)
    if metadata["dataset_id"] != download["dataset_id"]:
        raise s3_demo.DemoError("Trained model does not reference the downloaded dataset.")
    views.say(views.training_counts(metadata))
    views.show(views.metrics_table(evaluation))

## Step 8 — Predict the Week 5 games

**What this does:** reloads the newly trained model and predicts every upcoming Week 5 game. The output is
P(home team wins), the probability that the home team wins: this is binary classification (win or lose) with a
probability output, not regression.

**Why it matters:** these games have not been played yet. The model uses last season's results only.

In [ ]:
with views.step(8, "prediction"):
    prediction_report = predict_saved_sample(RUN_DIR / "model", RUN_DIR / "dataset", RUN_DIR / "predictions",
                                             allow_retrospective_demo=True)
    predictions = list(csv.DictReader(io.StringIO(
        read_file(RUN_DIR / "predictions" / "predictions.csv").decode("utf-8"))))
    views.say(f"**{prediction_report['count']} upcoming Week {WEEK} games**, sorted by kickoff:")
    views.show(views.predictions_table(predictions))

## Step 8b — Try a hypothetical matchup

**What this does:** asks the same trained model about an invented matchup described only by the three features
in `HYPOTHETICAL` (Step 0).

**Why it matters:** it shows how the features move the probability. This is not a real game.

In [ ]:
with views.step(8, "hypothetical prediction"):
    model, _ = load_trusted_model(RUN_DIR / "model", allow_retrospective_demo=True)
    hypothetical = predict_vector(model, dict(zip(PREDICTORS, HYPOTHETICAL, strict=True)))
    views.say(views.hypothetical_sentence(HYPOTHETICAL, hypothetical["home_win_probability"]))

## Step 9 — Limitations and next steps

- **Prior-season features only.** The model uses last season's results, so it does not yet see current-season form,
  injuries, or roster changes.
- **Retrospective demo, not an approved live forecast.** We cannot yet verify when each piece of historical data was
  first published, so results stay labeled as a retrospective demonstration.
- **Next steps:** in-season features, a sportsbook benchmark (The Odds API), drift monitoring (alerts when incoming
  data or accuracy changes over time), and serverless automation (scheduled cloud runs without a server to manage).